# USDJPY data explorer

Loads the MT5 export with `data_loader.load_ohlcv` and shows it as tables. Run all cells with **Run All** at the top.

In [1]:
import pandas as pd
from backtest_engine.data_loader import MT5_SERVER_TZ, load_ohlcv, resample_ohlcv, trading_day

PATH = r"C:\Users\hp\OneDrive\Desktop\MT5\USDJPY.csv"

FMT = {"open": "{:.3f}", "high": "{:.3f}", "low": "{:.3f}", "close": "{:.3f}",
       "volume": "{:,.0f}", "spread": "{:.1f}"}

def show(df, caption, daily=False):
    """Nicely formatted table: readable timestamps, fixed decimals.

    daily=True labels each bar by its trading day (17:00-17:00 New York)."""
    df = df.copy()
    if daily:
        df.insert(0, "opens (UTC)", df.index.strftime("%m-%d %H:%M"))
        df.index = [f"{d:%a %Y-%m-%d}" for d in trading_day(df.index)]
        df.index.name = "trading day"
    else:
        df.index = df.index.strftime("%Y-%m-%d %H:%M")
        df.index.name = "time (UTC)"
    return df.style.format(FMT).set_caption(caption)

# MT5 times are broker server time (UTC+2); convert them to real UTC.
df = load_ohlcv(PATH, tz=MT5_SERVER_TZ)
print(f"Loaded {len(df):,} bars from {df.index[0]:%Y-%m-%d %H:%M} to {df.index[-1]:%Y-%m-%d %H:%M}")

Loaded 100,000 bars from 2025-12-17 14:59 to 2026-03-27 10:56


## First and last 1-minute bars

In [2]:
show(df.head(10), "First 10 bars (1 min)")

,open,high,low,close,volume,spread
time (UTC),,,,,,
2025-12-17 14:59,155.465,155.493,155.465,155.487,40,4.0
2025-12-17 15:00,155.486,155.502,155.481,155.481,38,3.0
2025-12-17 15:01,155.477,155.481,155.464,155.472,28,4.0
2025-12-17 15:02,155.471,155.499,155.463,155.499,50,4.0
2025-12-17 15:03,155.499,155.525,155.499,155.511,45,3.0
2025-12-17 15:04,155.512,155.515,155.488,155.506,42,3.0
2025-12-17 15:05,155.507,155.507,155.468,155.473,46,3.0
2025-12-17 15:06,155.469,155.497,155.459,155.460,42,3.0
2025-12-17 15:07,155.459,155.514,155.454,155.504,51,3.0


In [3]:
show(df.tail(10), "Last 10 bars (1 min)")

,open,high,low,close,volume,spread
time (UTC),,,,,,
2026-03-27 10:47,159.930,159.936,159.925,159.928,50,3.0
2026-03-27 10:48,159.928,159.940,159.919,159.922,52,3.0
2026-03-27 10:49,159.922,159.922,159.907,159.910,65,3.0
2026-03-27 10:50,159.910,159.919,159.907,159.907,59,3.0
2026-03-27 10:51,159.908,159.918,159.902,159.904,91,3.0
2026-03-27 10:52,159.904,159.913,159.901,159.902,49,3.0
2026-03-27 10:53,159.902,159.911,159.889,159.895,53,3.0
2026-03-27 10:54,159.895,159.905,159.890,159.896,72,3.0
2026-03-27 10:55,159.897,159.921,159.897,159.908,51,3.0


## Summary statistics

In [4]:
df.describe().style.format("{:,.3f}").set_caption("Summary statistics")

,open,high,low,close,volume,spread
count,"100,000.000","100,000.000","100,000.000","100,000.000","100,000.000","100,000.000"
mean,156.702,156.714,156.689,156.702,57.492,4.370
std,1.905,1.904,1.906,1.905,68.433,9.692
min,152.107,152.126,152.093,152.093,1.000,0.000
25%,155.635,155.646,155.625,155.636,26.000,3.000
50%,156.825,156.835,156.816,156.826,38.000,3.000
75%,158.232,158.244,158.219,158.232,56.000,4.000
max,159.971,159.976,159.964,159.971,"1,115.000",327.000


## Higher timeframes

In [5]:
h1 = resample_ohlcv(df, "1h")
show(h1.tail(10), "Last 10 bars, 1 hour")

,open,high,low,close,volume,spread
time (UTC),,,,,,
2026-03-27 01:00,159.560,159.595,159.453,159.480,"3,630",4.3
2026-03-27 02:00,159.479,159.583,159.455,159.525,"2,320",4.4
2026-03-27 03:00,159.526,159.624,159.501,159.614,"1,739",4.6
2026-03-27 04:00,159.614,159.642,159.552,159.620,"1,660",4.6
2026-03-27 05:00,159.621,159.649,159.573,159.625,"2,436",2.9
2026-03-27 06:00,159.626,159.767,159.575,159.749,"2,439",2.7
2026-03-27 07:00,159.749,159.976,159.737,159.946,"4,058",2.5
2026-03-27 08:00,159.947,159.954,159.751,159.838,"4,317",2.5
2026-03-27 09:00,159.839,159.941,159.764,159.868,"3,705",2.5


In [6]:
# Daily bars run 17:00-17:00 New York time (the forex day), so there is no Sunday bar.
d1 = resample_ohlcv(df, "1D")
show(d1.tail(10), "Last 10 daily bars (17:00-17:00 New York)", daily=True)

,opens (UTC),open,high,low,close,volume,spread
trading day,,,,,,,
Mon 2026-03-16,03-15 21:00,159.571,159.750,158.848,159.002,"252,642",0.3
Tue 2026-03-17,03-16 21:00,159.002,159.497,158.722,158.944,"128,345",2.1
Wed 2026-03-18,03-17 21:00,158.944,159.896,158.567,159.801,"101,335",3.8
Thu 2026-03-19,03-18 21:00,159.801,159.854,157.508,157.691,"149,013",3.1
Fri 2026-03-20,03-19 21:00,157.690,159.387,157.607,159.224,"109,123",3.7
Mon 2026-03-23,03-22 21:00,159.276,159.656,158.019,158.405,"184,803",1.5
Tue 2026-03-24,03-23 21:00,158.405,159.189,158.257,158.660,"134,361",3.7
Wed 2026-03-25,03-24 21:00,158.659,159.500,158.564,159.427,"105,998",4.6
Thu 2026-03-26,03-25 21:00,159.426,159.842,159.285,159.771,"84,376",5.0
